In [3]:
!pip -q install pyspark

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, count, date_format

spark = SparkSession.builder \
    .appName("Retail Store Level Insights") \
    .getOrCreate()

print("Spark session created successfully!")

Spark session created successfully!


In [5]:
sales_df = spark.read.csv(
    "/content/sales.csv",
    header=True,
    inferSchema=True
)

products_df = spark.read.csv(
    "/content/products.csv",
    header=True,
    inferSchema=True
)

print("Sales Data")
sales_df.show(5)

print("Products Data")
products_df.show(5)

Sales Data
+-------+--------+----------+--------+-----+----+--------+
|sale_id|store_id|product_id|quantity|price|cost|discount|
+-------+--------+----------+--------+-----+----+--------+
|      1|     101|       201|      10|  500| 300|      20|
|      2|     101|       202|       5| 1000| 650|      50|
|      3|     102|       201|       8|  500| 300|      25|
|      4|     102|       203|       4| 1500|1000|     100|
|      5|     103|       202|      12| 1000| 650|      75|
+-------+--------+----------+--------+-----+----+--------+
only showing top 5 rows
Products Data
+----------+------------+-----------+
|product_id|product_name|   category|
+----------+------------+-----------+
|       201|      Laptop|Electronics|
|       202|  Headphones|Accessories|
|       203|    Keyboard|Accessories|
+----------+------------+-----------+



In [6]:
print("Sales Columns:")
sales_df.printSchema()

print("Products Columns:")
products_df.printSchema()

Sales Columns:
root
 |-- sale_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- cost: integer (nullable = true)
 |-- discount: integer (nullable = true)

Products Columns:
root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)



In [7]:
product_sales = sales_df.groupBy("product_id").agg(
    sum("quantity").alias("total_quantity_sold"),
    sum(col("quantity") * col("price")).alias("total_revenue")
)

product_insights = product_sales.join(
    products_df,
    on="product_id",
    how="left"
)

underperforming_products = product_insights.filter(
    col("total_quantity_sold") < 5
)

underperforming_products.show(truncate=False)

+----------+-------------------+-------------+------------+--------+
|product_id|total_quantity_sold|total_revenue|product_name|category|
+----------+-------------------+-------------+------------+--------+
+----------+-------------------+-------------+------------+--------+



In [9]:
total_store_revenue = sales_df.groupBy(
    "store_id"
).agg(
    sum(col("quantity") * col("price")).alias("total_revenue")
)

total_store_revenue.orderBy(
    "store_id"
).show(truncate=False)

+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|101     |21500        |
|102     |17000        |
|103     |21000        |
+--------+-------------+



In [11]:
store_summary = sales_df.groupBy(
    "store_id"
).agg(
    avg(col("quantity") * col("price")).alias("average_transaction_value"),
    count("*").alias("transactions_recorded")
)

store_summary.show(truncate=False)

+--------+-------------------------+---------------------+
|store_id|average_transaction_value|transactions_recorded|
+--------+-------------------------+---------------------+
|101     |5375.0                   |4                    |
|103     |7000.0                   |3                    |
|102     |5666.666666666667        |3                    |
+--------+-------------------------+---------------------+



In [13]:
underperforming_products.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/content/underperforming_products")

total_store_revenue.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/content/total_store_revenue")

store_summary.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/content/store_summary")

print("All Week 3 output files saved successfully!")

All Week 3 output files saved successfully!
